## Libraries

In [401]:
# pip install rapidfuzz

In [402]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rapidfuzz import process, utils


## Import Dataset

In [403]:
#Import the training dataset
train = pd.read_csv('../original_datasets/train.csv')
train.head()

,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0


In [404]:
#Import the test dataset
test = pd.read_csv('../original_datasets/test.csv')

## Functions

We will use *functions* to facilitate our writting. All our functions will be stored in this section for easier readability. 

All functions will have their definitions.


In [405]:
#Lists with the correct values for the features


correct_transmission = ['Manual', 'Semi-Auto', 'Automatic', 'Other','Unknown']
correct_fuelType = ['Petrol', 'Diesel', 'Hybrid', 'Electric', 'Other', 'Unknown']
correct_brand = ['VW', 'Toyota', 'Audi', 'Ford', 'Skoda', 'Opel', 'Mercedes', 'Hyundai', 'BMW', 'Unknown']

#Apply rapidfuzz to the features
def clean_strings(x, l):
    """
    Standardizes a single string value by using fuzzy matching against a list of correct values.

    It handles missing values, normalizes the input string, and attempts to find the 
    closest match in the provided list. If the match score is below a threshold (80), 
    or if the input is missing, the value is categorized as 'Unknown'.

    Args:
        x (str or object): The input string value (a single cell from a DataFrame column).
        l (list): The list of correct, allowed string values (e.g., correct_transmission).

    Returns:
        str: The standardized string value, which will be one of the values from the 
             correct list (l) or 'Unknown'.
    """
    if pd.isna(x):
        return 'Unknown'
    x = x.strip().title()
    match, score, _ = process.extractOne(x, l, processor=utils.default_process)    
    if score > 70:
        return match
    else:
        return 'Unknown'

In [406]:
def export_file(df, filename):
    pd.to_csv(f'new_datasets/{filename}', index=False)

## New Features

We will create new features, that we think that might help the model predict better the test dataset. This features are:

- **mileage_per_year**: We get how much the car is driven, on average, a year
- **tax_engineSize**: Some larger engines might have higher taxes, so, multiplying them, can exposes us a relationship the model find useful.
- **age_mileage**: With this feature we can get a extimation on the condition of the car  

In [407]:
test['mileage_per_year'] = test['mileage'] / (2025 - test['year'])
train['mileage_per_year'] = train['mileage'] / (2025 - train['year'])

In [408]:
test['tax_engineSize'] = test['tax'] *  test['engineSize']
train['tax_engineSize'] = train['tax'] * train['engineSize']

In [409]:
test['age_mileage'] = (2025 - test['year']) *  test['mileage']
train['age_mileage'] = (2025 - train['year']) * train['mileage']

In [410]:
test['age'] = 2025 - test['year']
train['age'] = 2025 - train['year']

## Inconsistent Data

Has we seen in the Data Understanding file, there is some features that have problems in their data:
- Negative values
- Typos 

#### Negative Data

For the negative data, we will use the *absolute values* to transform the incorrect data. First, we will create a list with the features that have these type of problem, and then, we use the method **abs()** to get the absolute value. Since the method doesn't get information of other rows in the dataset, we don't need to worry about leakage.

In [411]:
#Features where exists negavite data 
negative_features = ['mileage', 'mpg', 'engineSize', 'previousOwners', 'tax', 'mileage_per_year', 'tax_engineSize', 'age_mileage']

In [412]:
#Get the absolute value
train[negative_features] = train[negative_features].abs()
test[negative_features] = test[negative_features].abs()

In [413]:
train[negative_features].min()

mileage             1.000000
mpg                 1.100000
engineSize          0.000000
previousOwners      0.000000
tax                 0.000000
mileage_per_year    0.041667
tax_engineSize      0.000000
age_mileage         5.000000
dtype: float64

#### Typos

Also, as seen before, some categorical features have typos. To solve this problem we will use the library *rapidfuzz*. Rapizfuzz is a string matching library that will calculate the similarities between the words (the one in the list created and the ones from the dataset). For more efficient coding, we created a function (**clean_strings**) to apply the changes to the train and test dataset for the selected features.

##### Transmission

In [414]:
for df in [train, test]:
    df['transmission'] = df['transmission'].apply(lambda x: clean_strings(x, correct_transmission))

In [415]:
train['transmission'].value_counts()

transmission
Manual       41627
Semi-Auto    16872
Automatic    15211
Unknown       2258
Other            5
Name: count, dtype: int64

##### FuelType

In [416]:
for df in [train, test]:
    df['fuelType'] = df['fuelType'].apply(lambda x: clean_strings(x, correct_fuelType))

In [417]:
train['fuelType'].value_counts()

fuelType
Petrol      41181
Diesel      30885
Hybrid       2225
Unknown      1511
Other         167
Electric        4
Name: count, dtype: int64

##### Brand

In [418]:
for df in [train, test]:
    df['Brand'] = df['Brand'].apply(lambda x: clean_strings(x, correct_brand))

In [419]:
train['Brand'].value_counts()

Brand
Ford        16063
Mercedes    11674
VW          10385
Opel         9352
BMW          7392
Audi         7325
Toyota       4622
Skoda        4303
Hyundai      3336
Unknown      1521
Name: count, dtype: int64

## Outliers

In [420]:
train = train[train['year'] <= 2020]

## Export

In [ ]:
# Removemos 'carID' porque não é feature preditiva
# Mantemos 'price' fora do X_trainval
y_trainval = train['price']
X_trainval = train.drop(columns=['price', 'carID', 'paintQuality%', 'year', 'model'])

# Removemos 'carID' mas guardamo-lo se precisarmos dele para submissão (embora o Modeling o carregue do original)
X_test_final = test.drop(columns=['carID', 'paintQuality%', 'year', 'model'])

# Estes ficheiros vão conter Strings (ex: "Ford") e NaNs. É normal!
X_trainval.to_csv('../new_datasets/X_trainval_preprocessed.csv', index=False)
y_trainval.to_csv('../new_datasets/y_trainval.csv', index=False)
X_test_final.to_csv('../new_datasets/X_test_preprocessed.csv', index=False)

print("Dados exportados com sucesso! Prontos para o Loop Manual.")

Dados exportados com sucesso! Prontos para o Loop Manual.
